# 06·02 — BLEU & ROUGE: Reference-Based Evaluation

> **Module:** 06 – LLM Evaluation  
> **Notebook:** 2 of 4  
> **Objective:** Understand, implement, and critically evaluate BLEU and ROUGE — the workhorses of automated NLP evaluation — and know precisely when they fail.

---

## 🎯 Learning Objectives

1. Derive BLEU and ROUGE from first principles — every formula, every design choice
2. Implement both metrics from scratch without using libraries
3. Visualize where each metric agrees and disagrees with human judgment
4. Identify the specific failure modes that make them poor for modern LLMs
5. Know which task+metric pairs are still valid in 2024

---

## 1. The Reference-Based Evaluation Paradigm

Both BLEU and ROUGE operate on the same assumption:

```
IF a system output overlaps with a human-written reference → it's probably good

Hypothesis:  "The cat sat on the mat."
Reference:   "A cat was sitting on the mat."

Overlap:     ["cat", "on", "the", "mat"] → 4 matching words
```

This works well for tasks with **constrained correct outputs** (translation, summarization).
It breaks for tasks with **many valid outputs** (open-ended generation, QA, chat).

---

## 2. BLEU: Bilingual Evaluation Understudy

**Origin:** Designed in 2002 by Papineni et al. for machine translation evaluation.
The key insight: good translations share n-grams with human reference translations.

### 2.1 Modified N-gram Precision

Simple precision has a problem — a system that just repeats "the" 100 times gets high precision:

```
Hypothesis: "the the the the the"
Reference:  "The cat sat on the mat"
Precision = 5/5 = 1.0   ← WRONG, this is terrible output
```

**Modified precision** clips each n-gram count by its max count in ANY reference:

```
Hypothesis: "the the the the the"
Reference:  "The cat sat on the mat"   ("the" appears 1 time)
Clipped count of "the" = min(5, 1) = 1
Modified precision = 1/5 = 0.2   ← Much more reasonable
```

### 2.2 Brevity Penalty

Short outputs can game precision — "cat" matches perfectly:

```
BP = 1                        if len(hyp) > len(ref)
BP = exp(1 - len(ref)/len(hyp)) if len(hyp) ≤ len(ref)
```

### 2.3 Final BLEU Score

```
BLEU-N = BP · exp( Σₙ wₙ · log pₙ )

Where:
  pₙ  = modified n-gram precision for n-grams of size n
  wₙ  = weight (usually 1/N for uniform weighting)
  BP  = brevity penalty
  N   = 4 (BLEU-4 is the standard)
```

BLEU-4 uses n-grams from 1 to 4, geometric mean of their precisions.

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install nltk rouge-score evaluate datasets matplotlib seaborn -q

import re
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
from typing import List, Dict, Tuple, Optional
from itertools import chain

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Setup complete.")

## 3. BLEU: From-Scratch Implementation

In [ ]:
def tokenize(text: str) -> List[str]:
    """Simple whitespace + punctuation tokenizer."""
    return re.findall(r'\b\w+\b', text.lower())


def get_ngrams(tokens: List[str], n: int) -> Counter:
    """Extract n-gram counts from a token list."""
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1))


def modified_precision(hypothesis: List[str], references: List[List[str]], n: int) -> Tuple[int, int]:
    """
    Compute clipped n-gram precision.

    Returns (clipped_count, total_hyp_ngrams) — not the ratio,
    because BLEU aggregates counts across the corpus before dividing.
    """
    hyp_ngrams = get_ngrams(hypothesis, n)

    if not hyp_ngrams:
        return 0, 0

    # Max count of each n-gram across ALL references
    max_ref_counts = Counter()
    for ref in references:
        ref_ngrams = get_ngrams(ref, n)
        for ngram, count in ref_ngrams.items():
            max_ref_counts[ngram] = max(max_ref_counts[ngram], count)

    # Clip hypothesis counts
    clipped = sum(
        min(count, max_ref_counts.get(ngram, 0))
        for ngram, count in hyp_ngrams.items()
    )
    total = sum(hyp_ngrams.values())
    return clipped, total


def brevity_penalty(hypothesis_len: int, reference_len: int) -> float:
    """Penalises hypotheses that are shorter than the reference."""
    if hypothesis_len >= reference_len:
        return 1.0
    return math.exp(1 - reference_len / hypothesis_len)


def bleu_score(
    hypothesis: str,
    references: List[str],
    max_n: int = 4,
    weights: Optional[List[float]] = None,
    smooth: bool = True,
) -> Dict:
    """
    Compute BLEU score for a single hypothesis against one or more references.

    Args:
        hypothesis:  System output string
        references:  List of reference strings (usually 1-4 human translations)
        max_n:       Maximum n-gram order (BLEU-4 is standard)
        weights:     N-gram weights (default: uniform 1/N)
        smooth:      Apply smoothing for zero-count n-grams (recommended)

    Returns:
        dict with 'bleu', 'precisions', 'brevity_penalty', 'length_ratio'
    """
    hyp_tokens  = tokenize(hypothesis)
    ref_tokens  = [tokenize(r) for r in references]

    if weights is None:
        weights = [1.0 / max_n] * max_n

    # Best reference length (closest to hypothesis length)
    ref_len = min(
        (abs(len(hyp_tokens) - len(r)), len(r))
        for r in ref_tokens
    )[1]
    hyp_len = len(hyp_tokens)

    # Compute per-n precision
    precisions = []
    for n in range(1, max_n + 1):
        clipped, total = modified_precision(hyp_tokens, ref_tokens, n)
        if total == 0:
            precisions.append(0.0)
        elif clipped == 0:
            # Smoothing: add epsilon to avoid log(0)
            precisions.append(1.0 / (total * (2 ** n)) if smooth else 0.0)
        else:
            precisions.append(clipped / total)

    # Weighted geometric mean of precisions
    log_avg = sum(w * math.log(p) for w, p in zip(weights, precisions) if p > 0)
    bp = brevity_penalty(hyp_len, ref_len)
    bleu = bp * math.exp(log_avg) if any(p > 0 for p in precisions) else 0.0

    return {
        "bleu":           round(bleu, 4),
        "bleu_pct":       round(bleu * 100, 2),
        "precisions":     [round(p, 4) for p in precisions],
        "brevity_penalty": round(bp, 4),
        "length_ratio":   round(hyp_len / ref_len, 3) if ref_len > 0 else 0,
        "hyp_len":        hyp_len,
        "ref_len":        ref_len,
    }


# ── Sanity checks ──────────────────────────────────────────────────────────
ref = ["The cat sat on the mat"]

test_pairs = [
    ("Perfect match",                "The cat sat on the mat"),
    ("Good translation",             "The cat is sitting on the mat"),
    ("Different but valid",          "A cat rested on the mat"),
    ("Partial overlap",              "The dog sat on the floor"),
    ("Too short (BP penalized)",     "Cat mat"),
    ("Repetition exploit",           "the the the the the the"),
    ("Completely wrong",             "I love ice cream very much"),
]

print(f"Reference: {ref[0]}")
print()
print(f"{'Case':<30} {'BLEU':>6} {'P1':>6} {'P2':>6} {'P3':>6} {'P4':>6} {'BP':>6}")
print("-" * 70)
for case_name, hyp in test_pairs:
    r = bleu_score(hyp, ref)
    prec = r['precisions']
    print(f"{case_name:<30} {r['bleu_pct']:>6.2f} "
          f"{prec[0]:>6.3f} {prec[1]:>6.3f} {prec[2]:>6.3f} {prec[3]:>6.3f} "
          f"{r['brevity_penalty']:>6.3f}")

## 4. ROUGE: Recall-Oriented Understudy for Gisting Evaluation

**Origin:** Lin (2004), designed for automatic summarization evaluation.  
Key difference from BLEU: **ROUGE emphasizes recall** — does the summary contain the important content?

```
BLEU:  Precision-focused  →  "Does the hypothesis contain reference n-grams?"
ROUGE: Recall-focused     →  "Does the hypothesis cover what's in the reference?"

ROUGE-N  =  n-gram recall
ROUGE-L  =  longest common subsequence (captures word order without n-gram rigidity)
ROUGE-W  =  weighted LCS (rewards consecutive matches)
```

The **F1 variant** combines precision and recall (used in most modern evaluations):
```
F1 = 2 · Precision · Recall / (Precision + Recall)
```

In [ ]:
def rouge_n(
    hypothesis: str,
    reference: str,
    n: int = 2,
) -> Dict[str, float]:
    """
    Compute ROUGE-N (precision, recall, F1) for n-grams of size n.

    ROUGE-1 (unigram) measures content coverage.
    ROUGE-2 (bigram) measures fluency + content.
    """
    hyp_ngrams = get_ngrams(tokenize(hypothesis), n)
    ref_ngrams = get_ngrams(tokenize(reference),  n)

    if not ref_ngrams or not hyp_ngrams:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    # Count overlapping n-grams
    overlap = sum(
        min(hyp_ngrams[ng], ref_ngrams[ng])
        for ng in hyp_ngrams
        if ng in ref_ngrams
    )

    precision = overlap / sum(hyp_ngrams.values())
    recall    = overlap / sum(ref_ngrams.values())
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1, 4),
        "overlap":   overlap,
    }


def lcs_length(x: List, y: List) -> int:
    """Compute length of the Longest Common Subsequence via DP."""
    m, n = len(x), len(y)
    # Space-optimized: only keep two rows
    prev = [0] * (n + 1)
    curr = [0] * (n + 1)
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            curr[j] = prev[j-1] + 1 if x[i-1] == y[j-1] else max(curr[j-1], prev[j])
        prev, curr = curr, [0] * (n + 1)
    return prev[n]


def rouge_l(
    hypothesis: str,
    reference: str,
    beta: float = 1.2,  # β > 1 weighs recall more than precision
) -> Dict[str, float]:
    """
    Compute ROUGE-L using the Longest Common Subsequence.

    LCS captures word order implicitly without requiring contiguous matching.
    Example:
      Hyp: "police killed the gunman"
      Ref: "police kill the suspect"
      LCS: ["police", "the"]  →  length 2 (even though no bigram matches!)
    """
    hyp_tokens = tokenize(hypothesis)
    ref_tokens = tokenize(reference)

    if not hyp_tokens or not ref_tokens:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    lcs_len = lcs_length(hyp_tokens, ref_tokens)

    precision = lcs_len / len(hyp_tokens)
    recall    = lcs_len / len(ref_tokens)

    denom = (precision + beta**2 * recall)
    f1 = ((1 + beta**2) * precision * recall / denom) if denom > 0 else 0.0

    return {
        "precision": round(precision, 4),
        "recall":    round(recall, 4),
        "f1":        round(f1, 4),
        "lcs_length": lcs_len,
    }


def rouge_score_all(hypothesis: str, reference: str) -> Dict:
    """Compute ROUGE-1, ROUGE-2, and ROUGE-L together."""
    return {
        "rouge1": rouge_n(hypothesis, reference, n=1),
        "rouge2": rouge_n(hypothesis, reference, n=2),
        "rougeL": rouge_l(hypothesis, reference),
    }


# ── Demo ──────────────────────────────────────────────────────────────────
reference_text = (
    "The cat sat on the mat and looked at the dog sleeping peacefully near the window."
)

summaries = [
    ("Near-perfect",          "A cat sat on the mat watching the dog sleep near the window."),
    ("Good but verbose",      "There was a cat on the mat. It was looking at the dog which was sleeping. The dog was near the window."),
    ("Too short (high prec, low rec)",  "Cat on mat."),
    ("Too long (low prec, high rec)",   "The cat sat on the mat and looked at the dog sleeping peacefully near the window and also did many other things."),
    ("Paraphrase (no overlap)",         "A feline rested on the rug observing a slumbering canine by the windowsill."),
    ("Completely wrong",                "I ate pizza for dinner last night at the restaurant downtown."),
]

print(f"Reference: {reference_text}")
print()
print(f"{'Case':<35} {'R1-F1':>7} {'R2-F1':>7} {'RL-F1':>7} {'R1-P':>6} {'R1-R':>6}")
print("-" * 72)
rouge_data = []
for case, hyp in summaries:
    r = rouge_score_all(hyp, reference_text)
    rouge_data.append((case, r))
    print(f"{case:<35} "
          f"{r['rouge1']['f1']:>7.3f} "
          f"{r['rouge2']['f1']:>7.3f} "
          f"{r['rougeL']['f1']:>7.3f} "
          f"{r['rouge1']['precision']:>6.3f} "
          f"{r['rouge1']['recall']:>6.3f}")

In [ ]:
# Visualize Precision vs Recall trade-off
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
for case, r in rouge_data:
    p = r['rouge1']['precision']
    rec = r['rouge1']['recall']
    f1 = r['rouge1']['f1']
    ax.scatter(p, rec, s=150, zorder=5)
    ax.annotate(case[:20], (p, rec), textcoords='offset points', xytext=(5, 3), fontsize=8)

# F1 iso-curves
p_grid = np.linspace(0.01, 1, 100)
for f1_target in [0.2, 0.4, 0.6, 0.8]:
    r_grid = f1_target * p_grid / (2 * p_grid - f1_target + 1e-9)
    mask = (r_grid >= 0) & (r_grid <= 1)
    ax.plot(p_grid[mask], r_grid[mask], '--', alpha=0.3, color='gray', linewidth=1)
    ax.annotate(f'F1={f1_target}', (p_grid[mask][-1], r_grid[mask][-1]),
                fontsize=8, color='gray')

ax.set_xlabel('ROUGE-1 Precision', fontsize=12)
ax.set_ylabel('ROUGE-1 Recall', fontsize=12)
ax.set_title('ROUGE-1 Precision vs Recall Trade-off', fontsize=12)
ax.set_xlim(-0.05, 1.1)
ax.set_ylim(-0.05, 1.1)
ax.grid(alpha=0.3)

ax = axes[1]
metrics = ['ROUGE-1', 'ROUGE-2', 'ROUGE-L']
x = np.arange(len(summaries))
w = 0.25
for i, metric in enumerate(metrics):
    key = ['rouge1', 'rouge2', 'rougeL'][i]
    scores = [r[key]['f1'] for _, r in rouge_data]
    ax.bar(x + i*w, scores, width=w, label=metric, alpha=0.85)

ax.set_xticks(x + w)
ax.set_xticklabels([c[:15] for c, _ in rouge_data], rotation=30, ha='right', fontsize=9)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title('ROUGE-1, ROUGE-2, ROUGE-L Comparison', fontsize=12)
ax.legend()
ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('../figures/06_rouge_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Experiment: Correlation with Human Judgments

The most important question: **do BLEU/ROUGE actually correlate with human quality assessments?**

In [ ]:
# Simulated human judgments dataset
# In practice, use WMT human evaluation data or SummEval benchmark
# These examples illustrate real phenomena documented in the literature

mt_examples = [
    {
        "source":    "Der schnelle braune Fuchs springt über den faulen Hund.",
        "reference": "The quick brown fox jumps over the lazy dog.",
        "hypothesis":"The quick brown fox jumps over the lazy dog.",
        "human_score": 5.0,   # Perfect
        "note": "perfect_match",
    },
    {
        "source":    "Der schnelle braune Fuchs springt über den faulen Hund.",
        "reference": "The quick brown fox jumps over the lazy dog.",
        "hypothesis":"The swift brown fox leaps over the lethargic hound.",
        "human_score": 4.5,   # Excellent — same meaning, different words
        "note": "paraphrase_high_quality",
    },
    {
        "source":    "Der schnelle braune Fuchs springt über den faulen Hund.",
        "reference": "The quick brown fox jumps over the lazy dog.",
        "hypothesis":"A quick brown fox is jumping over a lazy dog.",
        "human_score": 4.0,   # Good
        "note": "minor_changes",
    },
    {
        "source":    "Der schnelle braune Fuchs springt über den faulen Hund.",
        "reference": "The quick brown fox jumps over the lazy dog.",
        "hypothesis":"The lazy fox jumps over the quick dog.",
        "human_score": 2.5,   # Poor — word order matters for meaning
        "note": "word_order_error",
    },
    {
        "source":    "Der schnelle braune Fuchs springt über den faulen Hund.",
        "reference": "The quick brown fox jumps over the lazy dog.",
        "hypothesis":"The quick brown fox.",
        "human_score": 2.0,   # Bad — incomplete
        "note": "incomplete",
    },
    {
        "source":    "Das Wetter in Berlin ist heute sonnig.",
        "reference": "The weather in Berlin is sunny today.",
        "hypothesis":"The weather in Berlin is sunny today.",
        "human_score": 5.0,
        "note": "perfect_match_2",
    },
    {
        "source":    "Das Wetter in Berlin ist heute sonnig.",
        "reference": "The weather in Berlin is sunny today.",
        "hypothesis":"Today it is a beautiful sunny day in the city of Berlin in Germany.",
        "human_score": 3.5,   # Over-generated, but correct meaning
        "note": "over_generated",
    },
    {
        "source":    "Das Wetter in Berlin ist heute sonnig.",
        "reference": "The weather in Berlin is sunny today.",
        "hypothesis":"The the the the the the the sunny today",
        "human_score": 0.5,   # Garbage
        "note": "garbage_output",
    },
]

# Compute metrics and compare with human scores
computed = []
for ex in mt_examples:
    b = bleu_score(ex['hypothesis'], [ex['reference']])
    r = rouge_score_all(ex['hypothesis'], ex['reference'])
    computed.append({
        'note': ex['note'],
        'human': ex['human_score'],
        'bleu': b['bleu_pct'],
        'rouge1': r['rouge1']['f1'],
        'rouge2': r['rouge2']['f1'],
        'rougeL': r['rougeL']['f1'],
    })

print(f"{'Note':<28} {'Human':>6} {'BLEU':>6} {'R1-F1':>7} {'R2-F1':>7} {'RL-F1':>7}")
print("-" * 65)
for c in computed:
    print(f"{c['note']:<28} {c['human']:>6.1f} {c['bleu']:>6.1f} "
          f"{c['rouge1']:>7.3f} {c['rouge2']:>7.3f} {c['rougeL']:>7.3f}")

# Correlation
from scipy import stats
human_scores = [c['human'] for c in computed]
for metric in ['bleu', 'rouge1', 'rouge2', 'rougeL']:
    scores = [c[metric] for c in computed]
    corr, pval = stats.pearsonr(human_scores, scores)
    spear, _ = stats.spearmanr(human_scores, scores)
    print(f"{metric.upper():<8}: Pearson r={corr:.3f}, Spearman ρ={spear:.3f}")

In [ ]:
# Visualize the correlation
fig, axes = plt.subplots(1, 4, figsize=(16, 5))

human_scores = [c['human'] for c in computed]
metric_keys = ['bleu', 'rouge1', 'rouge2', 'rougeL']
metric_names = ['BLEU', 'ROUGE-1', 'ROUGE-2', 'ROUGE-L']
colors_scatter = plt.cm.RdYlGn(np.array(human_scores) / 5)

for ax, key, name in zip(axes, metric_keys, metric_names):
    scores = [c[key] for c in computed]
    corr, _ = stats.pearsonr(human_scores, scores)

    ax.scatter(human_scores, scores, c=colors_scatter, s=120, zorder=5, edgecolors='gray')
    for c in computed:
        ax.annotate(c['note'][:10], (c['human'], c[key]),
                    textcoords='offset points', xytext=(3, 3), fontsize=6.5)

    # Trend line
    m, b = np.polyfit(human_scores, scores, 1)
    x_range = np.linspace(min(human_scores), max(human_scores), 50)
    ax.plot(x_range, m*x_range + b, 'r--', alpha=0.6)

    ax.set_xlabel('Human Score (0-5)', fontsize=10)
    ax.set_ylabel(name, fontsize=10)
    ax.set_title(f'{name}\nPearson r = {corr:.2f}', fontsize=11, fontweight='bold')
    ax.grid(alpha=0.3)

plt.suptitle('Automatic Metrics vs Human Judgments', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../figures/06_human_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. The Paraphrase Problem: The Core Limitation

In [ ]:
# Demonstrate the fundamental flaw: semantically equivalent paraphrases score VERY differently

ref = ["Scientists have discovered a new treatment that significantly reduces cancer cell growth."]

paraphrases = [
    # These should ALL get similar scores — same meaning, different words
    ("Original reference (score should = 1.0)",
     "Scientists have discovered a new treatment that significantly reduces cancer cell growth."),

    ("Synonym substitution",
     "Researchers have found a novel therapy that substantially inhibits tumor cell proliferation."),

    ("Passive → active voice",
     "A new cancer treatment that markedly reduces tumor growth has been discovered by scientists."),

    ("Different sentence structure",
     "The cancer cell growth is significantly reduced by a newly discovered treatment."),

    ("Nominalization",
     "The discovery of a new cancer therapy promises significant reduction of tumor growth."),

    # This is slightly WRONG but will score higher due to surface overlap
    ("⚠️ Subtly wrong (negation)",
     "Scientists have discovered a new treatment that does NOT reduce cancer cell growth."),
]

print("The Paraphrase Problem — Same Meaning, Very Different Scores:")
print(f"Reference: {ref[0]}")
print()
print(f"{'Variation':<40} {'BLEU':>7} {'ROUGE-1':>8} {'ROUGE-2':>8} {'ROUGE-L':>8}")
print("-" * 75)
for label, hyp in paraphrases:
    b = bleu_score(hyp, ref)
    r = rouge_score_all(hyp, ref[0])
    print(f"{label:<40} {b['bleu_pct']:>7.1f} "
          f"{r['rouge1']['f1']:>8.3f} "
          f"{r['rouge2']['f1']:>8.3f} "
          f"{r['rougeL']['f1']:>8.3f}")

print()
print("⚠️  Note: 'Subtly wrong (negation)' scores HIGHER than valid paraphrases!")
print("    BLEU/ROUGE cannot detect semantic errors — they only see surface overlap.")
print("    This is precisely why LLM-as-judge and task-specific metrics are needed.")

## 7. When to Use (and Not Use) These Metrics

### Decision Framework

In [ ]:
# Visualize: metric applicability by task type
tasks = [
    "Machine Translation",
    "Text Summarization",
    "Question Answering",
    "Code Generation",
    "Open-ended Chat",
    "Instruction Following",
    "Creative Writing",
    "Factual QA",
]
metrics = ["BLEU", "ROUGE", "METEOR", "BERTScore", "LLM-judge"]

# Scores: 0=not applicable, 1=weak, 2=moderate, 3=good
applicability = [
#   BLEU  RGE  METEOR BERTScore LLMjudge
    [3,    2,    3,     3,         2],  # MT
    [2,    3,    2,     3,         3],  # Summarization
    [1,    2,    2,     2,         3],  # QA
    [2,    1,    1,     2,         3],  # Code
    [0,    0,    1,     1,         3],  # Chat
    [0,    1,    1,     2,         3],  # Instruction
    [0,    1,    1,     1,         3],  # Creative writing
    [1,    1,    1,     2,         3],  # Factual QA
]

fig, ax = plt.subplots(figsize=(12, 7))
data = np.array(applicability)

colors = ['#d73027', '#fc8d59', '#fee090', '#91cf60', '#1a9850']
from matplotlib.colors import ListedColormap
cmap = ListedColormap(['#fee0d2', '#fc9272', '#fb6a4a', '#41ab5d'])

im = ax.imshow(data, cmap=cmap, vmin=0, vmax=3, aspect='auto')

ax.set_xticks(range(len(metrics)))
ax.set_yticks(range(len(tasks)))
ax.set_xticklabels(metrics, fontsize=12, fontweight='bold')
ax.set_yticklabels(tasks, fontsize=11)

labels = {0: 'N/A', 1: 'Weak', 2: 'OK', 3: 'Good'}
for i in range(len(tasks)):
    for j in range(len(metrics)):
        val = data[i, j]
        ax.text(j, i, labels[val], ha='center', va='center',
                fontsize=10, fontweight='bold',
                color='white' if val >= 2 else 'black')

ax.set_title('Metric Applicability by Task Type\n(N/A = not suitable, Good = recommended)', 
             fontsize=13, fontweight='bold')

plt.colorbar(im, ax=ax, label='Applicability', ticks=[0, 1, 2, 3],
             fraction=0.02, pad=0.02)
plt.tight_layout()
plt.savefig('../figures/06_metric_applicability.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Engineering Notes

### Summary: BLEU vs ROUGE

| Aspect | BLEU | ROUGE |
|--------|------|-------|
| **Orientation** | Precision | Recall |
| **Primary task** | Machine translation | Summarization |
| **Multiple references** | Supported natively | F1 averages across refs |
| **Brevity handling** | Brevity penalty | Recall naturally penalises short output |
| **Word order** | Via n-grams (indirect) | ROUGE-L via LCS |
| **Paraphrase sensitivity** | High (fails on paraphrases) | High (same failure) |
| **Corpus vs sentence level** | Best at corpus level | Works at sentence level |

### Production Usage Guidelines

```python
# Use HuggingFace `evaluate` library in production — it handles
# edge cases and matches official implementations
import evaluate

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

# Always tokenize consistently!
bleu_result = bleu.compute(
    predictions=["The cat sat on the mat"],
    references=[["The cat is on the mat"]]
)

rouge_result = rouge.compute(
    predictions=["The cat sat on the mat"],
    references=["The cat is on the mat"]
)
```

### Modern Alternatives

| Metric | How | When to use |
|--------|-----|-------------|
| **BERTScore** | Semantic similarity via BERT embeddings | When paraphrases are valid |
| **METEOR** | BLEU + synonyms + stemming | MT with morphology-rich languages |
| **COMET** | Learned MT metric (very human-correlated) | High-quality MT evaluation |
| **LLM-as-judge** | GPT-4 / Claude grades the output | Open-ended, instruction-following tasks |

## 9. Exercises

1. **Corpus BLEU**: Implement corpus-level BLEU (aggregate counts across all sentences before computing precision). Compare it to averaging sentence-level BLEU. Why do they differ?

2. **METEOR**: METEOR extends BLEU with synonym matching via WordNet. Implement METEOR and show how it handles the paraphrase examples that fool BLEU.

3. **SummEval Correlation**: Download the SummEval dataset (human judgments on CNN/DailyMail summaries). Compute ROUGE-1/2/L and measure their Spearman correlation with human coherence, consistency, fluency, and relevance scores.

4. **WMT Human Evaluation**: Access WMT translation human evaluation data and compare BLEU correlation vs BERTScore correlation with human direct assessments.

5. **Reference Sensitivity**: Take 5 MT hypotheses, have 3 different people write references, and compute BLEU/ROUGE against each individual reference vs all 3 combined. How much does the choice of reference affect the score?

---

## 10. References

- [Papineni et al. (2002) — BLEU: a Method for Automatic Evaluation of Machine Translation](https://aclanthology.org/P02-1040/)
- [Lin (2004) — ROUGE: A Package for Automatic Evaluation of Summaries](https://aclanthology.org/W04-1013/)
- [Mathur et al. (2020) — Tangled up in BLEU: Reevaluating the Evaluation of Automatic Machine Translation Evaluation Metrics](https://arxiv.org/abs/2006.06264)
- [Bhandari et al. (2020) — Re-evaluating Evaluation in Text Summarization](https://arxiv.org/abs/2010.07100)
- [HuggingFace Evaluate Library](https://huggingface.co/docs/evaluate)